# Setup LangSmith API
Retrievals can be traced here for easier debugging.

In [8]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../../')))
import ollama
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_API_KEY'] = '123-123-213-123-123-123'

![LangGraph Flow](../../langgraph%20designs/graph_design_v1.png)

# Graph States

In [ ]:
from typing_extensions import TypedDict
from typing import Optional, List, Dict, Any

class GraphState(TypedDict):
    # Core user input
    text_query: str
    image_path: Optional[str]  # Path to uploaded image, if any

    # Routing/intent
    query_type : str # currently being divide into 'emergency'/'Q&A'/'irrelevant'.
    
    # Q&A path
    refined_query: Optional[str]
    queries_for_retrieval: Optional[List[str]]
    current_sub_query: Optional[str]
    retrieved_docs: Optional[List[Dict[str, Any]]]  # Results from retrieval
    reranked_docs: Optional[List[Dict[str, Any]]]   # After rerank step

    # Feedback loop
    followup_questions: Optional[List[str]]
    user_responses: Optional[List[str]]
    loop_count: int
    hypotheses: Optional[List[str]]  # Current working hypotheses
    next_action: Optional[str]       # What the agent plans to do next ("ask_user", "retrieve", "final_answer", etc.)
    pending_question: Optional[str]  # If the agent wants to ask the user something
    pending_action: Optional[str]    # If the agent wants to perform a tool/action
    user_actions: Optional[List[str]] # Actions the user has taken (e.g., "smelled ear", "provided photo")
    intermediate_thoughts: Optional[List[str]] # Chain-of-thought or reasoning steps

    # Answer generation
    generated_answer: Optional[str]
    hallucination_check: Optional[bool]
    answer_sufficient: Optional[bool]

    # Emergency path
    emergency_instructions: Optional[str]
    emergency_retrieved_docs: Optional[List[Dict[str, Any]]]

    # Web search
    web_search_results: Optional[List[Dict[str, Any]]]

    # Final output
    final_answer: Optional[str]

    # Misc/trace/debug
    path_taken: Optional[List[str]]
    error: Optional[str]

<h1> Graph Nodes

## Query Handler Node 

Before LLM analyze user query and image, it will be assessed with "Is this veterinary-related?". This will ensure our AI tool will not be used for other purpose.

In [33]:
def query_handler(state):
    text_query = state.get("text_query", "")
    image_path = state.get("image_path", None)

    prompt = (
        "You are a domain classifier for a veterinary assistant. "
        "If an image is provided, understand the image from veterinary point of view."
        "A user query is the combination of text query and image(if there is). "
        "Then, classify the user query into one of three categories:\n"
        "1. 'emergency' — If the user query is about a veterinary emergency (e.g., mass bleeding, serious bone fracture, unconsciousness, severe breathing difficulty, or other life-threatening situations).\n"
        "2. 'Q&A' — If the user query is about is about general veterinary questions, symptom checks, or non-emergency animal health issues.\n\n"
        "3. 'irrelevant' — If the user query is NOT about veterinary, animal health, pet care, etc.\n"
        "Your response must be exactly one of: 'irrelevant', 'emergency', or 'Q&A'. Do not explain your answer or add anything else.\n\n"
        f"User input: {text_query}\n"
    )

    messages = [{
        "role": "user",
        "content": prompt,
        "images": []
    }]

    if image_path and os.path.exists(image_path):
        messages[0]["images"].append(image_path)

    response = ollama.chat(
        model="minicpm-v:8b",
        messages=messages,
        options={"temperature": 0.2}
    )
    result = response['message']['content'].strip().lower()
    # Only allow the three valid outputs
    if result not in ['irrelevant', 'emergency', 'q&a']:
        result = 'irrelevant'
    
    return {"query_type": result}

### test

In [34]:
def test_query_handler_node(query_handler, test_query, image_path=None):
    # Build the initial state
    state = {
        "text_query": test_query,
        "image_path": image_path
    }
    # Call the query handler node
    new_state = query_handler(state)
    # Print the results
    print("Input Query:", test_query)
    if image_path:
        print("Image Path:", image_path)
    print("Updated State:", new_state)
    print("Query Type:", new_state.get("query_type", "N/A"))
    print("-" * 40)

# --- Example usage ---
test_query_handler_node(query_handler, "What vaccines does my cat need?")
test_query_handler_node(query_handler, "My cat is bleeding a lot after being hit by a car.")
test_query_handler_node(query_handler, "How do I fix my car engine?")
test_query_handler_node(query_handler, "What should I do?", image_path="../emergency_cat.jpg")
test_query_handler_node(query_handler, "What should I feed to this cat?", image_path="../skinny_cat.jpg")


Input Query: What vaccines does my cat need?
Updated State: {'text_query': 'What vaccines does my cat need?', 'image_path': None, 'query_type': 'q&a'}
Query Type: q&a
----------------------------------------
Input Query: My cat is bleeding a lot after being hit by a car.
Updated State: {'text_query': 'My cat is bleeding a lot after being hit by a car.', 'image_path': None, 'query_type': 'emergency'}
Query Type: emergency
----------------------------------------
Input Query: How do I fix my car engine?
Updated State: {'text_query': 'How do I fix my car engine?', 'image_path': None, 'query_type': 'irrelevant'}
Query Type: irrelevant
----------------------------------------
Input Query: What should I do?
Image Path: ../emergency_cat.jpg
Updated State: {'text_query': 'What should I do?', 'image_path': '../emergency_cat.jpg', 'query_type': 'emergency'}
Query Type: emergency
----------------------------------------
Input Query: What should I feed to this cat?
Image Path: ../skinny_cat.jpg
Up

# Q&A Path

## Query Refinement

In [70]:
def get_image_summary(image_path):
    prompt = """From a feline veterinary stand point, provide a highly detailed and objective 
                description of the image. Focus on all observable elements, actions, 
                objects, subjects, their attributes (e.g., color, size, texture), 
                their spatial relationships, and any discernible context or implied scene. 
                Also focus on all possible health issue.
                Describe any text present in the image. This description must be exhaustive 
                and purely factual, capturing every significant visual detail to serve as a 
                comprehensive textual representation for further analysis by another AI model. 
                If the image is entirely irrelevant or contains no discernible subject, 
                state "No relevant visual information."""
    messages = [{
        "role": "user",
        "content": prompt,
        "images": [image_path]
    }]
    response = ollama.chat(
        model="minicpm-v:8b",
        messages=messages,
        options={"temperature": 0.2}
    )
    return response['message']['content']

def query_refinement_node(state):
    text_query = state.get("text_query", "")
    image_path = state.get("image_path", None)
    image_summary = get_image_summary(image_path) if image_path else ""

    if image_summary:
        prompt = (
        "You are a veterinary assistant AI. Your task is to rewrite and expand the user's question about their cat to make it more effective for searching a veterinary knowledge base.\n\n"
        "You are NOT being asked to give medical advice, make a diagnosis, or recommend treatments.\n\n"
        "Use the image description only to clarify the concern, but **do not invent or assume** any details (such as environment, causes, or severity) not explicitly mentioned by the user or image.\n\n"
        "The refined query must:\n"
        "- Accurately represent the user's concern about their cat\n"
        "- Factually describe any symptoms or visible signs\n"
        "- Include open-ended questions about **possible causes**, **diagnostic steps**, and **general management or prevention**\n"
        "- Avoid assumptions, conclusions, or overly specific scenarios\n"
        "- Be phrased as **a single paragraph**, clear and concise, suitable for search retrieval\n"
        "- Do not add new symptoms, behaviors, or environmental details unless they appear in the user query or image description.\n"
        "- Output **only** the refined query — no introductions or explanations\n\n"
        "Here are examples:\n"
        "---\n"
        "User query: My cat keeps shaking her head a lot.\n"
        "Image description: Redness and dark wax visible in one ear.\n"
        "Refined query: My cat has been shaking her head frequently, and I've noticed redness and dark wax in one ear. I'd like to understand what might be causing these symptoms, what diagnostic steps are typically used to evaluate ear conditions in cats, and what general management or preventive options may apply.\n"
        "---\n"
        "User query: My cat has been throwing up for two days.\n"
        "Image description: Pile of partially digested food on carpet.\n"
        "Refined query: My cat has been vomiting for the past two days, with piles of partially digested food. I want to explore potential causes of vomiting in cats, how to tell if it's serious, what diagnostic approaches are used, and general advice for managing this before seeing a vet.\n"
        "---\n"
        f"User query: {text_query}\n"
        f"Image description: {image_summary}\n"
        "Refined query:"
    )
    else:
        prompt = (
        "You are a veterinary assistant AI. Your task is to rewrite and expand the user's question about their cat to make it more effective for searching a veterinary knowledge base.\n\n"
        "You are NOT being asked to give medical advice, make a diagnosis, or recommend treatments.\n\n"
        "The refined query must:\n"
        "- Clearly describe the user's concern about their cat\n"
        "- Remain **neutral and open-ended**, avoiding assumptions or conclusions\n"
        "- Include helpful questions about **possible causes**, **diagnostic considerations**, and **general management or prevention**\n"
        "- Be phrased as **a single paragraph**, clear and concise, with no extra fluff\n"
        "Do not add new symptoms, behaviors, or environmental details unless they appear in the user query or image description."
        "- Output **only** the refined query — no introductions or explanations\n\n"
        "Here are examples:\n"
        "---\n"
        "User query: I think my cat has a fever, her nose is hot.\n"
        "Refined query: I'm concerned my cat may have a fever because her nose feels hotter than usual. I want to understand what can cause fever in cats, what signs to look for, and what general steps I should take before consulting a veterinarian.\n"
        "---\n"
        "User query: My cat's been sleeping more than usual and not eating.\n"
        "Refined query: My cat is sleeping much more than usual and has lost interest in eating. I'd like to know what potential causes could lead to these symptoms, how to assess if it's urgent, and what general steps I can take before visiting a vet.\n"
        "---\n"
        f"User query: {text_query}\n"
        "Refined query:"
    )

    messages = [{
        "role": "user",
        "content": prompt
    }]
    response = ollama.chat(
        model="llama3.1:8b", 
        messages=messages,
        options={"temperature": 0}
    )
    return {"refined_query": response['message']['content']}

### test

In [72]:
def test_query_refinement_node(query_refinement_node, test_query, image_path=None):
    global test_refined_query
    # Build the initial state
    state = {
        "text_query": test_query,
        "image_path": image_path
    }
    # Call the query refinement node
    new_state = query_refinement_node(state)
    test_refined_query = new_state.get("refined_query", "N/A")
    # Print the results
    print("Input Query:", test_query)
    if image_path:
        print("Image Path:", image_path)
    print("Refined Query:", new_state.get("refined_query", "N/A"))
    print("-" * 40)

# --- Ears Chapter Example ---
#test_query_refinement_node(query_refinement_node, "What happened to my cat ear? It's being it for a long time. Sometimes I even see blood and wounds in its ear. ", image_path="../cat_ear_problem.jpeg")
#test_query_refinement_node(query_refinement_node, "My cat has being scratching its ear too often. There are some dark greasy thing in it. It sratch its ear so often and so hard that I see wounds and blood in it. What should I do?")

# --- Emergency and Infecious Disease Chapter Example ---
test_query_refinement_node(query_refinement_node, "I think my cat is having a heat stroke, what should I do?")

Input Query: I think my cat is having a heat stroke, what should I do?
Refined Query: I'm concerned that my cat may be experiencing heat-related distress. She has been panting excessively and seems lethargic. I'd like to understand the possible causes of heat stress in cats, how to assess its severity, and what general steps I can take to help her recover before consulting a veterinarian.
----------------------------------------


## Query Decomposition

In [ ]:
import ollama
import json
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser


def query_decomposition(state):
    refined_query = state['refined_query']

    query_decomposition_prompt = ChatPromptTemplate.from_template(
    """You are a veterinary knowledge assistant. Your task is to break down the following refined query into a list of **concise, semantically rich phrases**, each representing a **different aspect** of the query.

    Guidelines:
    - Each phrase should represent a **distinct, relevant concept** or question implied in the user's concern.
    - Focus on **causes, symptoms, diagnostics, management steps, risk factors, and context**.
    - Avoid repeating or overlapping ideas.
    - Do NOT include general fluff or irrelevant topics.
    - Use phrases or noun-like expressions (not full sentences).
    - Aim for maximum relevance to the original refined query — each phrase should help retrieve part of a comprehensive answer.
    - At the end, include 2-3 **visually grounded search phrases** that require images, diagrams, or visual aids to understand.

    Output only a **JSON array of strings**, with no extra text or explanation.

    Refined query: {refined_query}
    """
    )

    # Create the query decomposition chain
    query_decomposition_chain = (
        query_decomposition_prompt  
        | ChatOllama(model="llama3.1:8b")  
        | JsonOutputParser() 
    )

    # --- Demonstration of query decomposition ---

    print(f"Original refined query: {refined_query[:300]} ....")

    decomposed_queries = query_decomposition_chain.invoke({"refined_query": refined_query})
    # Try to extract the JSON array from the response

    print("-" * 80)
    # print(f"Decomposed queries:\n{decomposed_queries}")

    print(f"There are {len(decomposed_queries)} queries after decomposition \n")

    return {"queries_for_retrieval": decomposed_queries}

### Test

In [78]:
def test_query_decomposition(query_decomposition_func, refined_query):
    global test_decomposed_queries
    # Build the initial state
    state = {
        "refined_query": refined_query
    }
    # Call the query decomposition function
    new_state = query_decomposition_func(state)
    test_decomposed_queries = new_state['queries_for_retrieval']
    # Print the results
    print("Decomposed Sub-Queries:")
    print(new_state['queries_for_retrieval'])

# --- Example usage ---
test_query_decomposition( query_decomposition, test_refined_query)

Original refined query: I'm concerned that my cat may be experiencing heat-related distress. She has been panting excessively and seems lethargic. I'd like to understand the possible causes of heat stress in cats, how to assess its severity, and what general steps I can take to help her recover before consulting a veterina ....
--------------------------------------------------------------------------------
There are 9 queries after decomposition 

Here's a example of the first one: Causes of heat stress in cats
Decomposed Sub-Queries:
['Causes of heat stress in cats', 'Symptoms of heat-related distress in felines', 'Panting as indicator of heat stress', 'Lethargy in cats due to heat exposure', 'Assessing severity of heat stress in cats', 'Risk factors for heat-related illness in cats', 'General management steps for heat-stressed cats', 'Providing relief for overheated cat at home', 'Preventing heat stroke in outdoor or indoor cats']


## Contextual Retrievals

Based on decomposed sub queries, we are able to retrieve contexutally close aligned Documents from the vector database. 

### Setup Unified Retriever (Retrieve text, table, images)

In [25]:
from langchain_experimental.open_clip import OpenCLIPEmbeddings
from langchain_chroma import Chroma
from unified_retriever import UnifiedRetriever
def init_retriever():

    persist_directory = '../../chroma/EmergencyInfectiveDisease'
    id_key = "doc_id"

    open_clip_embeddings = OpenCLIPEmbeddings(model_name="ViT-g-14", checkpoint="laion2b_s34b_b88k")

    # Vectorstore for summaries (for similarity search)
    vectorstore = Chroma(
        collection_name="summaries_and_images",
        persist_directory=persist_directory,
        embedding_function=open_clip_embeddings
    )
    # Persistent docstore for originals (all modalities)
    docstore = Chroma(
        collection_name="originals",
        persist_directory=persist_directory,
        embedding_function=open_clip_embeddings
    )

    retriever = UnifiedRetriever(vectorstore, docstore, id_key=id_key)
    return retriever

### Retrieval

In [82]:
seen_doc_ids = set()
all_results = []
retriever = init_retriever()

def contextual_retrieval_flat(state):
    seen_doc_ids = set()
    unique_docs = []
    retriever = init_retriever() 

    for query in state['queries_for_retrieval']:
        results = retriever.retrieve_multi_modal(query, k=3, )
        for res in results:
            doc_id = res.get('doc_id') or res.get('summary_metadata', {}).get('doc_id')
            if doc_id and doc_id not in seen_doc_ids:
                seen_doc_ids.add(doc_id)
                unique_docs.append(res)
    print(f"Total unique documents retrieved: {len(unique_docs)}")

    # No need to append new value to retrieved_docs, LangGraph's state reducer will handle it.
    return {"retrieved_docs": unique_docs}

### test

In [83]:
import copy

def test_contextual_retrieval(queries_for_retrieval):
    global test_retrived_doc

    test_state = {
        "queries_for_retrieval": queries_for_retrieval
    }

    # Use the flat contextual retrieval function
    new_state = contextual_retrieval_flat(test_state)
    unique_docs = new_state["retrieved_docs"]
    print("\nSample of unique retrieved docs:")
    for i, doc in enumerate(unique_docs):
        doc_id = doc.get('doc_id') or doc.get('summary_metadata', {}).get('doc_id')
        print(f"Doc {i}:")
        print(f"  Doc ID: {doc_id}")
        # Check if this is an image context doc
        if doc_id and doc_id.endswith('_context'):
            image_path = doc.get('summary_metadata', {}).get('image_path')
            print(f"  [IMAGE CONTEXT] Points to image file: {image_path}")
        print(f"  Type: {(doc.get('original_metadata') or {}).get('type')}")
        print(f"  Score: {doc.get('score')}")
        # print(f"  Summary: {doc.get('summary')[:100]}...")
        print(f" Original: {retriever.docstore._collection.get(ids=[doc_id], include=["documents"])}")
        print("-" * 40)
    print(f"Total unique docs retrieved: {len(unique_docs)}")
    
    test_retrived_doc = copy.deepcopy(unique_docs)

# Example usage:
test_contextual_retrieval(test_decomposed_queries)

Total unique documents retrieved: 11

Sample of unique retrieved docs:
Doc 0:
  Doc ID: 1d1c40cf-1369-4417-8792-aee3b3d5eabc
  Type: image
  Score: 1.1657415628433228
 Original: {'ids': ['1d1c40cf-1369-4417-8792-aee3b3d5eabc'], 'embeddings': None, 'documents': ['./figures/EmergencyInfectiveDisease/figure-3-5.jpg'], 'uris': None, 'included': ['documents'], 'data': None, 'metadatas': None}
----------------------------------------
Doc 1:
  Doc ID: 5626c0e6-4157-4b28-8c65-72a3d4813cde
  Type: image
  Score: 1.162426471710205
 Original: {'ids': ['5626c0e6-4157-4b28-8c65-72a3d4813cde'], 'embeddings': None, 'documents': ['./figures/EmergencyInfectiveDisease/figure-2-3.jpg'], 'uris': None, 'included': ['documents'], 'data': None, 'metadatas': None}
----------------------------------------
Doc 2:
  Doc ID: a936c6f7-f292-4aa2-a531-cb14a4251c0f
  Type: image
  Score: 1.1555404663085938
 Original: {'ids': ['a936c6f7-f292-4aa2-a531-cb14a4251c0f'], 'embeddings': None, 'documents': ['./figures/Emerge

## ReRank

Retrievals returns docs with high similarities based on cosine-similarity. However, we do need to re-rank their improtance on contexual level.

### Getting image, image_summary pair

In [ ]:
# truly multimodel [monoqwen], use here If running on Nvidia GPU Machine
# pip install "rerankers[monovlm]" qwen-vl-utils transformers
from rerankers import MonoQwen2VLReranker

def rerank_node_monoqwen(state):
    query = state['refined_query']
    candidates = state['retrieved_docs']

    # Prepare candidates for reranker
    rerank_inputs = []
    for doc in candidates:
        if doc.get("modality") == "text":
            rerank_inputs.append(doc["summary"])
        elif doc.get("modality") in ("image", "image_summary"):
            # Use image path if available, else fallback to summary
            image_path = doc.get("original_metadata", {}).get("image_path")
            if image_path:
                rerank_inputs.append(image_path)
            else:
                rerank_inputs.append(doc["summary"])
        else:
            rerank_inputs.append(doc["summary"])

    # Rerank
    from rerankers import MonoQwen2VLReranker
    reranker = MonoQwen2VLReranker.from_pretrained("Qwen/MonoQwen2-VL-v0.1")
    results = reranker.rerank(query, rerank_inputs, top_k=len(rerank_inputs))

    # Attach scores and sort
    for (idx, score) in results:
        candidates[idx]['rerank_score'] = float(score)
    reranked = sorted(candidates, key=lambda x: x.get('rerank_score', 0), reverse=True)
    return {"reranked_docs": reranked}

In [68]:
#Jina Reranker m0. GPU/CPU, but extremly slow in CPU
import base64
import os
from transformers import AutoModel


def image_to_base64(image_path):
    with open(image_path, "rb") as img_file:
        return base64.b64encode(img_file.read()).decode("utf-8")

def rerank_node_jina_vlm(state):
    query = state['refined_query']
    candidates = state['retrieved_docs']

    documents = []
    doc_types = []
    for doc in candidates:
        if doc.get("modality") in ("image", "image_summary"):
            image_path = doc.get("original_metadata", {}).get("image_path")
            if image_path and os.path.exists(image_path):
                documents.append(image_to_base64(image_path))
                doc_types.append("image")
            else:
                documents.append(doc["summary"])
                doc_types.append("text")
        else:
            documents.append(doc["summary"])
            doc_types.append("text")

    pairs = [[query, doc] for doc in documents]

    model = AutoModel.from_pretrained(
        'jinaai/jina-reranker-m0',
        torch_dtype="auto",
        trust_remote_code=True,
    )
    model.to('cpu')
    model.eval()

    # If most docs are images, use doc_type="image", else "text"
    n_images = doc_types.count("image")
    n_texts = doc_types.count("text")
    doc_type = "image" if n_images > n_texts else "text"

    # If mixed, filter and rerank separately, then merge (advanced)
    # For now, just use the dominant type
    scores = model.compute_score(pairs, max_length=2048, doc_type=doc_type)

    for doc, score in zip(candidates, scores):
        doc['rerank_score'] = float(score)
    reranked = sorted(candidates, key=lambda x: x.get('rerank_score', 0), reverse=True)
    return {"reranked_docs": reranked}

In [13]:
# Hybrid Method: CrossEncoder for text, VLM for image.
from sentence_transformers import CrossEncoder
import os
import ollama
def llm_image_relevance_score(query, image_path, image_summary=None):
    """
    Use Ollama (minicpm-v:8b) to rate the relevance of an image to the query.
    Passes the image file and, if available, the image summary.
    Returns a float score between 0 and 1.
    """
    prompt = f"""
    You are a veterinary assistant AI. Given the following user query and an image, rate how relevant the image is to answering the query.
    - User Query: "{query}"
    """
    if image_summary:
        prompt += f'- Image Summary: "{image_summary}"\n'
    prompt += "Respond with a single float between 0 (not relevant at all) and 1 (highly relevant). Only output the number, nothing else."

    try:
        messages = [{"role": "user", "content": prompt}]
        if image_path and os.path.exists(image_path):
            messages[0]["images"] = [image_path]
        response = ollama.chat(
            model="minicpm-v:8b",
            messages=messages,
            options={"temperature": 0.0}
        )
        content = response['message']['content'].strip()
        score = float(content.split()[0])
        score = max(0.0, min(1.0, score))
        return score
    except Exception as e:
        print(f"[llm_image_relevance_score] Error: {e}. Query: {query[:50]}... Image: {image_path}... Summary: {str(image_summary)[:50]}...")
        return 0.0

def rerank_node_hybrid_v2(state):
    query = state['refined_query']
    candidates = state['retrieved_docs']

    text_indices = []
    text_contents = []
    image_indices = []
    image_info = []

    for idx, doc in enumerate(candidates):
        modality = doc.get("modality") or (doc.get("original_metadata") or {}).get("type")
        if modality == "text":
            text_indices.append(idx)
            doc_id = (doc.get("original_metadata") or {}).get("doc_id") or doc.get("doc_id")
            # Fetch the original document from the docstore
            docstore = retriever.docstore
            original_text = None
            if doc_id and docstore:
                try:
                    original = docstore._collection.get(ids=[doc_id], include=["documents"])
                    original_text = original["documents"][0] if original["documents"] else None
                    print(f"for doc_id :{doc_id}, the original text: {original_text}")
                except Exception as e:
                    print(f"[rerank_node_hybrid_v2] Error fetching original text for doc_id {doc_id}: {e}")
            if not original_text:
                original_text = doc.get("summary", "")
            text_contents.append(original_text)
        elif modality == "image":
            # Use the image file for VLM
            image_path = (doc.get("original_metadata") or {}).get("image_path")
            image_summary = (doc.get("original_metadata") or {}).get("summary", "")
            image_indices.append(idx)
            image_info.append((image_path, image_summary if image_summary else None))
        elif modality == "image_summary":
            # Trace to the image file if possible
            image_path = (doc.get("original_metadata") or {}).get("image_path")
            image_summary = doc.get("summary", "")
            image_indices.append(idx)
            image_info.append((image_path, image_summary))
        else:
            # Fallback: treat as text
            text_indices.append(idx)
            text_contents.append(doc.get("summary", ""))

    # 1. Rerank text docs
    if text_contents:
        model = CrossEncoder("BAAI/bge-reranker-base")
        pairs = [(query, text) for text in text_contents]
        scores = model.predict(pairs)
        for idx, score in zip(text_indices, scores):
            candidates[idx]['rerank_score'] = float(score)

    # 2. Rerank images (and image summaries) with VLM
    for idx, (image_path, image_summary) in zip(image_indices, image_info):
        score = llm_image_relevance_score(query, image_path, image_summary)
        candidates[idx]['rerank_score'] = float(score)

    # 3. Sort all by rerank_score
    reranked = sorted(candidates, key=lambda x: x.get('rerank_score', 0), reverse=True)
    return {"reranked_docs": reranked}

## test

In [36]:
def test_rerank_node(rerank_node, refined_query, retrieved_docs, top_n=5):
    global reranked_docs

    state = {
      "refined_query": refined_query,
      "retrieved_docs": retrieved_docs,
      "docstore": retriever.docstore 
     }
    
    new_state = rerank_node(state)
    reranked_docs = new_state.get("reranked_docs", [])
    print(f"Total docs after reranking: {len(reranked_docs)}")
    print(f"Top {top_n} reranked docs (by rerank_score):")
    
    for i, doc in enumerate(reranked_docs[:top_n]):
        doc_id = doc.get('doc_id') or (doc.get('original_metadata') or {}).get('doc_id')
        modality = (doc.get('original_metadata') or {}).get('type') or doc.get('modality')
        print(f"Doc {i}:")
        print(f"  Doc ID: {doc_id}")
        print(f"  Type: {modality}")
        print(f"  Rerank Score: {doc.get('rerank_score')}")
        print(f"  Summary: {doc.get('summary')[:100]}...")
        print("-" * 40)

    scores = [doc.get('rerank_score') for doc in reranked_docs if doc.get('rerank_score') is not None]
    if scores and scores == sorted(scores, reverse=True):
        print("PASS: Docs are sorted by rerank_score descending.")
    else:
        print("FAIL: Docs are not sorted correctly or scores are missing.")

# Example usage:
refined_query = "What are possible causes and diagnostic steps for chronic ear infections or otitis in cats, characterized by visible brownish-orange debris, discharge, and wounds in the ear canal, potentially accompanied by signs of infection such as redness, swelling, and bleeding, and how can these conditions be distinguished from other potential health issues that may affect a cat's auditory system"
test_rerank_node(rerank_node_hybrid_v2, refined_query, test_retrived_doc)
reranked_docs

Total docs after reranking: 23
Top 5 reranked docs (by rerank_score):
Doc 0:
  Doc ID: 67e88179-5a0d-4d0a-bf9a-eefc9c4648f1_context
  Type: image_summary
  Rerank Score: 0.5
  Summary: The image shows a person gently handling and grooming a Siamese cat in what appears to be a quiet en...
----------------------------------------
Doc 1:
  Doc ID: 4f62c061-edb2-4061-aed3-7cc94e45306f
  Type: image
  Rerank Score: 0.25
  Summary: ./figures/EmergencyInfectiveDisease/figure-5-8.jpg...
----------------------------------------
Doc 2:
  Doc ID: ca48e12a-97b3-433c-851e-67109518c045
  Type: image
  Rerank Score: 0.25
  Summary: ./figures/EmergencyInfectiveDisease/figure-3-4.jpg...
----------------------------------------
Doc 3:
  Doc ID: a936c6f7-f292-4aa2-a531-cb14a4251c0f
  Type: image
  Rerank Score: 0.25
  Summary: ./figures/EmergencyInfectiveDisease/figure-2-2.jpg...
----------------------------------------
Doc 4:
  Doc ID: ca48e12a-97b3-433c-851e-67109518c045_context
  Type: image_summary
 

[{'modality': 'image_summary',
  'summary': 'The image shows a person gently handling and grooming a Siamese cat in what appears to be a quiet environment conducive for veterinary procedures such as bathing or medicating. The context emphasizes the importance of calmness, confidence, and gentle handling when dealing with cooperative cats during routine care tasks.',
  'original_metadata': {'doc_id': '67e88179-5a0d-4d0a-bf9a-eefc9c4648f1_context',
   'summary': 'The image shows a person gently handling and grooming a Siamese cat in what appears to be a quiet environment conducive for veterinary procedures such as bathing or medicating. The context emphasizes the importance of calmness, confidence, and gentle handling when dealing with cooperative cats during routine care tasks.',
   'image_path': './figures/EmergencyInfectiveDisease/figure-4-7.jpg',
   'type': 'image_summary'},
  'score': 1.3676557540893555,
  'doc_id': '67e88179-5a0d-4d0a-bf9a-eefc9c4648f1_context',
  'rerank_score': 0

In [37]:
def display_top_10_imgs_and_texts(reranked_docs, docstore):
    print("Top 10 Images and Original Texts:\n")
    count = 0
    for doc in reranked_docs:
        if count >= 10:
            break
        modality = (doc.get('original_metadata') or {}).get('type') or doc.get('modality')
        doc_id = doc.get('doc_id') or (doc.get('original_metadata') or {}).get('doc_id')
        score = doc.get('rerank_score')
        print(f"Doc {count}:")
        print(f"  Doc ID: {doc_id}")
        print(f"  Type: {modality}")
        print(f"  Rerank Score: {score}")
        if modality == "text":
            # Fetch original text from docstore
            original_text = None
            if doc_id and docstore:
                try:
                    original = docstore._collection.get(ids=[doc_id], include=["documents"])
                    original_text = original["documents"][0] if original["documents"] else None
                except Exception as e:
                    print(f"    [Error fetching original text for doc_id {doc_id}: {e}]")
            if not original_text:
                original_text = doc.get("summary", "")
            print("  Original Text:")
            print(f"    {original_text[:500]}{'...' if len(original_text) > 500 else ''}")
        elif modality in ("image", "image_summary"):
            image_path = (doc.get("original_metadata") or {}).get("image_path")
            print(f"  Image Path: {image_path}")
            print("  Image Summary:")
            print(f"    {doc.get('summary', '')[:500]}{'...' if len(doc.get('summary', '')) > 500 else ''}")
        else:
            print("  [Unknown modality]")
        print("-" * 60)
        count += 1

# Example usage:
display_top_10_imgs_and_texts(reranked_docs, retriever.docstore)

Top 10 Images and Original Texts:

Doc 0:
  Doc ID: 67e88179-5a0d-4d0a-bf9a-eefc9c4648f1_context
  Type: image_summary
  Rerank Score: 0.5
  Image Path: ./figures/EmergencyInfectiveDisease/figure-4-7.jpg
  Image Summary:
    The image shows a person gently handling and grooming a Siamese cat in what appears to be a quiet environment conducive for veterinary procedures such as bathing or medicating. The context emphasizes the importance of calmness, confidence, and gentle handling when dealing with cooperative cats during routine care tasks.
------------------------------------------------------------
Doc 1:
  Doc ID: 4f62c061-edb2-4061-aed3-7cc94e45306f
  Type: image
  Rerank Score: 0.25
  Image Path: ./figures/EmergencyInfectiveDisease/figure-5-8.jpg
  Image Summary:
    ./figures/EmergencyInfectiveDisease/figure-5-8.jpg
------------------------------------------------------------
Doc 2:
  Doc ID: ca48e12a-97b3-433c-851e-67109518c045
  Type: image
  Rerank Score: 0.25
  Image Path: ./

# Thinking Node

This step is to take all on-hand info and reranked doc to make analysis. Think about user's intent, what they want to know, what they need to know, also what AI need to know.

In [58]:
import re
def thinking_node(state):
    """
    Given the current state, use Qwen3 to reason step-by-step about how to answer the user's question.
    Updates state with intermediate thoughts, hypotheses, and next_action if more info or tools are needed.
    """
    user_query = state.get("text_query", "")
    image_summary = state.get("image_summary", "")
    retrieved_docs = state.get("reranked_docs") or state.get("retrieved_docs") or []
    prompt = (
        "You are a veterinary assistant AI. The user is a pet owner with little veterinary knowledge. "
        "Explain in simple, actionable language, only suggesting home-care steps. If the case is serious, remind the user to see a vet. "
        "Base your answer strictly on the provided docs. "
        "If you need more info, specify what and which tool to use. "
        "At the end, always choose one next action: [retrieve more info, ask the user a question, ready to answer].\n\n"

        "Actions:\n"
        "- retrieve more info: Output 'Next action: retrieve more info' and suggest new queries for the retriever.\n"
        "- ask the user a question: Output 'Next action: ask the user a question' and briefly explain why.\n"
        "- ready to answer: Output 'Next action: ready to answer'. If helpful, include up to 2 relevant images (provide their file paths from metadata).\n\n"

        "Respond in JSON, using one of these formats:\n"
        '{\n'
        '  \"thinking\": \"your reasoning\",\n'
        '  \"next_action\": \"retrieve more info\",\n'
        '  \"queries\": [\"query1\", \"query2\"],\n'
        '  \"user_response\": \"your answer\"\n'
        '}\n'
        "or\n"
        '{\n'
        '  \"thinking\": \"your reasoning\",\n'
        '  \"next_action\": \"ask the user a question\",\n'
        '  \"user_response\": \"your answer\"\n'
        '}\n'
        "or\n"
        '{\n'
        '  \"thinking\": \"your reasoning\",\n'
        '  \"next_action\": \"ready to answer\",\n'
        '  \"user_response\": \"your answer\",\n'
        '  \"images\": [\"image_path1\", \"image_path2\"]\n'
        '}\n\n'
        f"User question: {user_query}\n"
    )

    if image_summary:
        prompt += f"Image summary: {image_summary}\n"

    # Set how many to include
    TOP_TEXT = 3
    TOP_IMAGE = 2

    # Separate by modality
    text_docs = [doc for doc in retrieved_docs if (doc.get('modality') or (doc.get('original_metadata') or {}).get('type')) == 'text']
    image_docs = [doc for doc in retrieved_docs if (doc.get('modality') or (doc.get('original_metadata') or {}).get('type')) in ('image', 'image_summary')]

    # Sort by score (descending: higher is better for similarity)
    text_docs = sorted(text_docs, key=lambda d: d.get('score', 0), reverse=True)[:TOP_TEXT]
    image_docs = sorted(image_docs, key=lambda d: d.get('score', 0), reverse=True)[:TOP_IMAGE]

    prompt += "Relevant information from veterinary handbook:\n"
    for i, doc in enumerate(text_docs + image_docs):
        modality = doc.get('modality') or (doc.get('original_metadata') or {}).get('type')
        summary = doc.get('summary', '')
        if len(summary) > 400:
            summary = summary[:400] + '...'
        doc_id = doc.get('doc_id') or (doc.get('original_metadata') or {}).get('doc_id')
        if modality == 'image' or modality == 'image_summary':
            image_path = (doc.get('original_metadata') or {}).get('image_path')
            prompt += f"{i+1}. {modality}: [Image: {image_path}] {summary} (score: {doc.get('score')}, id: {doc_id})\n"
        else:
            prompt += f"{i+1}. {modality}: {summary} (score: {doc.get('score')}, id: {doc_id})\n"

    # Call Qwen3 via Ollama
    messages = [{"role": "user", "content": prompt}]
    response = ollama.chat(
        model="qwen3:8b",  # or another Qwen3 variant
        messages=messages,
        options={"temperature": 0.2},
    )
    llm_output = response['message']['content']

    think_match = re.search(r"<think>(.*?)</think>", llm_output, re.DOTALL | re.IGNORECASE)
    if think_match:
        reasoning = think_match.group(1).strip()
    else:
        reasoning = ""  # fallback if not found

    # Optionally, extract the rest (user-facing answer)
    user_response = re.sub(r"<think>.*?</think>", "", llm_output, flags=re.DOTALL | re.IGNORECASE).strip()

    print("✨Thinking Node✨: ", reasoning, "\n")
    print("✨User Response✨: ", user_response, "\n")

    state["intermediate_thoughts"] = state.get("intermediate_thoughts", []) + [reasoning]
    state["last_user_response"] = user_response  

    # Try to extract next_action from the output (simple heuristic, can be improved)
    if "retrieve more info" in llm_output.lower():
        state["next_action"] = "retrieve_more_info"
    elif "interpret a new image" in llm_output.lower() or "interpret new image" in llm_output.lower():
        state["next_action"] = "interpret_image"
    elif "ask the user" in llm_output.lower() or "ask user" in llm_output.lower():
        state["next_action"] = "ask_user"
    else:
        state["next_action"] = None

    # Optionally, extract hypotheses if the LLM lists them
    if "hypothesis" in llm_output.lower():
        state["hypotheses"] = [llm_output]

    return state

## Tools

In [ ]:
def book_retrieval_for_more(query, retriever, k=5):
    print("🔩Book Retriever Tool Called! 🔧")

def interpret_image(image_path, model = "minicpm-v:8b"):
    print("📷Image Interpreter Tool Called! ⛰️")


In [59]:
def test_thinking_node():
    """
    Test the thinking_node with a mock state including text query, optional image summary, and mixed retrieved_docs.
    Prints the updated state, intermediate thoughts, and next_action.
    """
    # Example mock state
    mock_state = {
        "text_query": test_refined_query,
        # Uncomment the next line to test with an image summary
        # "image_summary": "The image shows a cat's ear with visible dark debris and some redness.",
        "reranked_docs": reranked_docs
    }
    print("--- Before thinking_node ---")
    print({k: v for k, v in mock_state.items() if k != 'reranked_docs'})
    print("retrieved_docs count:", len(mock_state["reranked_docs"]))
    print()
    thinking_node(mock_state)


# Example usage:
test_thinking_node()

--- Before thinking_node ---
{'text_query': "I'm concerned that my cat may be experiencing heatstroke due to exposure to high temperatures, possibly exacerbated by lack of access to shade or water, and I'd like to know the possible causes, diagnostic considerations, such as monitoring vital signs and checking for signs of dehydration, and treatment options including providing a cool environment with fans or air conditioning, administering plenty of fresh water, and potentially using cooling pads or wet compresses, as well as any preventative measures to take in the future to prevent heatstroke, such as ensuring my home is climate-controlled, providing adequate shade and ventilation, and keeping an eye on my cat's behavior and physical condition during hot weather."}
retrieved_docs count: 23

✨Thinking Node✨:  Okay, let's tackle this user's question. They're worried their cat might have heatstroke due to high temps and lack of shade/water. They want to know causes, how to diagnose, trea